# Lezione 3 — Costruire una rete neurale

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 110 minuti**

## Obiettivi di apprendimento di questa lezione

- Costruire reti neurali di più strati con `nn.Sequential` e classi `nn.Module` personalizzate
- Scegliere funzioni di attivazione appropriate e capire perché sono necessarie
- Usare funzioni di perdita (`loss`) per misurare quanto male sta facendo il modello
- Scegliere un ottimizzatore (`optimizer`) per aggiornare i pesi in base al gradiente

Questo notebook accompagna il capitolo 03 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**.

## Parte 1 — Dalle reti minime alla pratica

Nella lezione 1 hai visto il modello più semplice possibile: una cella con un'unica entrata e un'unica uscita. In pratica, la maggior parte dei modelli hanno:

- Molte entrate (migliaia, milioni): i pixel di un'immagine, le parole di una frase, i sensori di un dispositivo IoT
- Molte uscite (da due a migliaia): le probabilità di ogni categoria in una classificazione
- Molti strati nascosti: questo è ciò che rende "profonda" la rete

Questo notebook ti guida nella costruzione di una rete che legge le immagini MNIST (28×28 pixel = 784 numeri di input) e predice la probabilità di ogni cifra (10 output). Sarai tu a decidere quanti strati nascosti aggiungere e quale funzione di attivazione usare.

## Parte 2 — Anatomia di una rete neurale multicapa

### Strati (layers) e attivazioni

Una rete neurale è una cascata di **strati lineari** e **funzioni di attivazione**:

1. Un **strato lineare** è una moltiplicazione per matrice + somma di bias: `output = input @ W + b`. Ogni peso `W` e `b` è un parametro che viene aggiornato durante l'addestramento.
2. Una **funzione di attivazione** introduce non linearità: `output_attivato = funzione_attivazione(output)`. Senza di essa, strati lineari apilati sarebbero ancora lineari, e il modello potrebbe solo imparare a fare previsioni rettilinee.

Due attivazioni comuni:

- **ReLU** (Rectified Linear Unit): `max(0, x)`. È semplice e veloce, e la più usata nei moderni modelli di deep learning.
- **Softmax**: trasforma gli output in probabilità che sommano a 1. Si usa solo nello strato finale per la classificazione.

### Forward pass

Un modello che legge un'immagine MNIST (784 input), ha 128 neuroni nascosti e predice 10 classi:

```
Input (784) -> Linear (784, 128) -> ReLU -> Linear (128, 10) -> Softmax -> Output (10 probabilità)
```

È il **forward pass**: dai dati di entrada fino alla previsione.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

print("Versión de PyTorch:", torch.__version__)
print("Versión de torchvision:", torchvision.__version__)

## Parte 3 — Costruire il modello con nn.Sequential

### Primo approccio: `nn.Sequential`

Il modo più veloce di impilare strati è usare `nn.Sequential`, che esegue gli strati in ordine.

In [ ]:
# Una capa totalmente conectada: 784 valores de entrada (28x28 píxeles aplanados), 64 de salida
capa_ejemplo = nn.Linear(in_features=784, out_features=64)

print("Forma de los pesos:", capa_ejemplo.weight.shape)  # 64 neuronas x 784 entradas
print("Forma de los sesgos:", capa_ejemplo.bias.shape)   # un sesgo por neurona

# Los pesos y sesgos se inicializan al azar: por eso rompemos la simetría entre neuronas
print("\nPrimeros valores de los pesos:\n", capa_ejemplo.weight[0, :5])

## Parte 4 — Funzione di perdita e ottimizzatore

Ora che hai il modello, hai bisogno di due cose per addestrarlo:

1. Una **funzione di perdita** (`loss function`) che misura "quanto male sta facendo il modello"
2. Un **ottimizzatore** (`optimizer`) che aggiorna i pesi in base al gradiente della perdita

### Funzione di perdita

Per un compito di classificazione multiclasse (come MNIST, con 10 classi), la norma è **Cross-Entropy Loss** (`nn.CrossEntropyLoss`):

```python
perdita = nn.CrossEntropyLoss()
```

`CrossEntropyLoss` si aspetta:
- **Logits** (output non trasformati della rete, senza Softmax)
- **Etichette** (i veri indici di classe, come il numero 7 per un'immagine che ritrae un sette)

Calcola automaticamente il Softmax dentro, quindi non devi aggiungerlo come strato finale.

### Ottimizzatore

Un **ottimizzatore** aggiorna i pesi usando i gradienti calcolati da `.backward()`. Quello più semplice e più usato è **SGD** (Stochastic Gradient Descent), ma **Adam** è più moderno e generalmente converge più velocemente:

```python
ottimizzatore = torch.optim.Adam(modello.parameters(), lr=0.001)
```

`lr` è il **learning rate**: un iperparametro che controlla quanto velocemente il modello apprende. Valori tipici sono tra 0.0001 e 0.1.

In [ ]:
valores_ejemplo = torch.tensor([-3.0, -0.5, 0.0, 1.2, 4.0])
print("Valores originales:", valores_ejemplo)
print("Tras aplicar ReLU:  ", F.relu(valores_ejemplo))

## Parte 5 — Il ciclo di addestramento

Ora che hai il modello, la perdita e l'ottimizzatore, il ciclo di addestramento di un'epoch è:

```
per ogni batch:
    1. Forward pass: dai il batch al modello
    2. Calcola la perdita
    3. backward(): calcola i gradienti
    4. ottimizzatore.step(): aggiorna i pesi
    5. ottimizzatore.zero_grad(): resetta i gradienti
```

La ragione del passaggio 5 è che `.backward()` **accumula** i gradienti: se non li resets, i gradienti del batch successivo si sommeranno ai precedenti.

In [ ]:
class ClasificadorMNIST(nn.Module):
    def __init__(self):
        # Llamada obligatoria al constructor de nn.Module
        super().__init__()

        # Declaramos las capas: cada nn.Linear conecta por completo
        # la capa anterior con la siguiente
        self.fc1 = nn.Linear(784, 128)  # entrada -> primera capa oculta
        self.fc2 = nn.Linear(128, 64)   # primera capa oculta -> segunda capa oculta
        self.fc3 = nn.Linear(64, 10)    # segunda capa oculta -> salida (10 dígitos)

    def forward(self, x):
        # ReLU tras cada capa oculta para introducir no linealidad
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # La salida no lleva ReLU: usamos log_softmax para obtener
        # una distribución de probabilidad sobre las 10 clases
        x = F.log_softmax(self.fc3(x), dim=1)
        return x


# Instanciamos la red: aquí se inicializan pesos y sesgos al azar
modelo = ClasificadorMNIST()
print(modelo)

## Parte 6 — Definire il modello con una classe personalizzata

Per modelli più complessi, invece di `nn.Sequential`, puoi definire una classe personalizzata ereditando da `nn.Module`. Questo ti dà più flessibilità: puoi avere logica condizionale, salti (skip connections), ecc.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transformacion = transforms.ToTensor()

datos_mnist = datasets.MNIST(root="./datos", train=True, download=True, transform=transformacion)
cargador = DataLoader(datos_mnist, batch_size=16, shuffle=True)

# Tomamos un único lote del cargador
imagenes, etiquetas = next(iter(cargador))
print("Forma del lote de imágenes:", imagenes.shape)   # [16, 1, 28, 28]
print("Forma del lote de etiquetas:", etiquetas.shape)  # [16]

## Riepilogo della lezione

- Una rete neurale è una cascata di strati lineari e funzioni di attivazione: niente attivazione = niente apprendimento di pattern complessi
- Per la classificazione multiclasse, usa `nn.CrossEntropyLoss` (che include Softmax interno) e non aggiungere Softmax come strato
- Un ottimizzatore aggiorna i pesi in base ai gradienti. `Adam` è una scelta moderna e robusta, `SGD` è più semplice
- Il ciclo di addestramento è: forward → perdita → backward → step dell'ottimizzatore → zero_grad
- Puoi usare `nn.Sequential` per reti semplici, o una classe personalizzata per più flessibilità

Nel notebook di pratica aggiungerai strati nascosti al modello, sperimenterai con learning rate diversi e osserverai come influiscono sulla velocità di convergenza.

In [ ]:
# Aplanamos cada imagen del lote: de [16, 1, 28, 28] a [16, 784]
# El -1 le dice a PyTorch "calcula tú este tamaño", aquí mantiene el número de imágenes del lote
imagenes_aplanadas = imagenes.view(-1, 784)
print("Forma tras aplanar:", imagenes_aplanadas.shape)

# El forward pass: los datos atraviesan la red y obtenemos una predicción
salida = modelo(imagenes_aplanadas)
print("Forma de la salida:", salida.shape)  # [16, 10]: una fila por imagen, 10 valores por fila

print("\nSalida para la primera imagen del lote:\n", salida[0])
print("\n¿Suman 1 las probabilidades de la primera imagen? (aproximadamente):",
      torch.exp(salida[0]).sum().item())

I valori di `salida` sono log-probabilità (per questo usiamo `torch.exp` per recuperare probabilità normali prima di sommarle): più alto è il valore per una classe, più la rete è sicura che l'immagine appartenga a quella cifra. Dato che non abbiamo ancora addestrato la rete, queste predizioni sono praticamente rumore: i pesi sono ancora i valori casuali iniziali.

## Parte 7 — Cosa fa autograd dietro le quinte

Anche se non lo vedi esplicitamente nel codice, ogni operazione del `forward` che hai appena eseguito (`self.fc1(x)`, `F.relu(...)`, `F.log_softmax(...)`) è stata registrata da **autograd**, il motore di differenziazione automatica di PyTorch. Quella registrazione forma quello che si chiama un **grafo di calcolo**: una mappa di quale operazione ha prodotto ciascun tensore e a partire da quali altri tensori.

Quel grafo è esattamente ciò di cui la lezione successiva avrà bisogno: quando calcoleremo quanto si sbaglia la rete (la perdita) e chiameremo `.backward()`, autograd percorrerà il grafo all'indietro per calcolare il gradiente di ogni peso e ogni bias, cioè in quale direzione e quanto bisogna aggiustarli per ridurre l'errore. Per ora, devi solo sapere che questo lavoro sta già accadendo in modo automatico ogni volta che esegui un `forward`; non devi programmarlo tu.

## Riepilogo della lezione

- Una rete neurale si organizza in strati (input, nascosti, output) formati da neuroni con pesi e bias
- Ogni rete in PyTorch eredita da `nn.Module`, con `__init__` per dichiarare gli strati e `forward` per collegarli
- `nn.Linear(in_features, out_features)` crea uno strato completamente connesso
- Le funzioni di attivazione (ReLU negli strati nascosti, `log_softmax` nell'output di un classificatore) introducono non linearità e rendono interpretabile il risultato
- Prima di passare un'immagine attraverso una rete di strati `nn.Linear`, bisogna appiattirla con `.view(-1, dimensione)`
- Il `forward` pass attiva già autograd, che registra il grafo di calcolo necessario per l'addestramento

Nella prossima lezione userai questa stessa architettura per addestrarla davvero: calcolare la perdita, propagarla all'indietro con `.backward()` e aggiornare i pesi con un optimizer.